<a href="https://colab.research.google.com/github/Tushika2024/MultiModal_AI/blob/main/0_datasetprep_audio_frame_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/multimodal_project/dataset"

os.chdir(PROJECT_DIR)

print(os.getcwd())

/content/drive/MyDrive/multimodal_project/dataset


In [ ]:
!ls -lh

total 4.3G
-rw------- 1 root root 4.3G May 30 11:57 archive.zip
drwx------ 2 root root 4.0K May 30 11:58 TrainValVideo


In [ ]:
!unzip -q archive.zip

In [ ]:
#after extraction to check folder structure
import os

for item in os.listdir():
    print(item)

.config
sample_data


In [ ]:
##checking videos in full datset
import os

video_dir = "/content/drive/MyDrive/multimodal_project/dataset/TrainValVideo"

videos = os.listdir(video_dir)

print("Number of videos:", len(videos))
print(videos[:10])

Number of videos: 5020
['video4614.mp4', 'video4610.mp4', 'video4588.mp4', 'video4611.mp4', 'video4638.mp4', 'video4637.mp4', 'video4623.mp4', 'video4647.mp4', 'video4640.mp4', 'video4628.mp4']


In [ ]:
#creating a subst
import os
import shutil

video_dir = "/content/drive/MyDrive/multimodal_project/dataset/TrainValVideo"

subset_dir = "subset_videos"

os.makedirs(subset_dir, exist_ok=True)

videos = sorted(os.listdir(video_dir))[:600]

for v in videos:
    shutil.copy(
        os.path.join(video_dir, v),
        os.path.join(subset_dir, v)
    )

print("Subset created")

Subset created


In [6]:
##checking videos in subset of 600 videos
import os

video_dir = "/content/drive/MyDrive/multimodal_project/dataset/subset_videos"

videos = os.listdir(video_dir)

print("Number of videos:", len(videos))
print(videos[:10])

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/multimodal_project/dataset/subset_videos'

In [ ]:
#create project strructure
import os

BASE = "/content/drive/MyDrive/multimodal_project"

folders = [
    "frames",
    "audio",
    "image_embeddings",
    "audio_embeddings",
    "video_embeddings",
    "checkpoints",
    "outputs"
]

for f in folders:
    os.makedirs(os.path.join(BASE, f), exist_ok=True)

print("Folders created")

Folders created


EMBEDDINGS

In [ ]:
!pip install -q opencv-python
!pip install -q transformers
!pip install -q librosa
!pip install -q moviepy
!pip install -q soundfile

In [5]:
#defining paths
import os

BASE_DIR = "/content/drive/MyDrive/multimodal_project"

VIDEO_DIR = os.path.join(
    BASE_DIR,
    "dataset",
    "subset_videos"
)

FRAME_DIR = os.path.join(
    BASE_DIR,
    "frames"
)

AUDIO_DIR = os.path.join(
    BASE_DIR,
    "audio"
)

IMAGE_EMB_DIR = os.path.join(
    BASE_DIR,
    "image_embeddings"
)

AUDIO_EMB_DIR = os.path.join(
    BASE_DIR,
    "audio_embeddings"
)

VIDEO_EMB_DIR = os.path.join(
    BASE_DIR,
    "video_embeddings"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs"
)

print("Video Count:",
      len(os.listdir(VIDEO_DIR)))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/multimodal_project/dataset/subset_videos'

In [ ]:
#extract middle frame
import cv2
import os

count = 0

for video in os.listdir(VIDEO_DIR):

    if not video.endswith(".mp4"):
        continue

    video_path = os.path.join(
        VIDEO_DIR,
        video
    )

    cap = cv2.VideoCapture(
        video_path
    )

    total_frames = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    middle_frame = total_frames // 2

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        middle_frame
    )

    success, frame = cap.read()

    if success:

        frame_name = video.replace(
            ".mp4",
            ".jpg"
        )

        cv2.imwrite(
            os.path.join(
                FRAME_DIR,
                frame_name
            ),
            frame
        )

        count += 1

    cap.release()

print(
    f"Frames Extracted: {count}"
)


Frames Extracted: 600


In [ ]:
print(
    "Frames:",
    len(os.listdir(FRAME_DIR))
)

Frames: 600


In [ ]:
#extract audio from videos
from moviepy.editor import VideoFileClip
import os

count = 0

for video in os.listdir(VIDEO_DIR):

    if not video.endswith(".mp4"):
        continue

    try:

        video_path = os.path.join(
            VIDEO_DIR,
            video
        )

        clip = VideoFileClip(
            video_path
        )

        audio_path = os.path.join(
            AUDIO_DIR,
            video.replace(
                ".mp4",
                ".wav"
            )
        )

        clip.audio.write_audiofile(
            audio_path,
            verbose=False,
            logger=None
        )

        count += 1

    except:

        pass

print(
    f"Audio Files: {count}"
)

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



Audio Files: 529


In [ ]:
#failed videos for audio extraction
import os

video_names = {
    f.replace(".mp4", "")
    for f in os.listdir(VIDEO_DIR)
}

audio_names = {
    f.replace(".wav", "")
    for f in os.listdir(AUDIO_DIR)
}

failed = sorted(video_names - audio_names)

print("Failed:", len(failed))
print(failed[:20])

Failed: 71
['video0', 'video1006', 'video1008', 'video101', 'video1045', 'video106', 'video1066', 'video1072', 'video1092', 'video1097', 'video1102', 'video1105', 'video1115', 'video1116', 'video1118', 'video1127', 'video1128', 'video1141', 'video1152', 'video1155']


In [ ]:
#verify audios
print(
    "Audio:",
    len(os.listdir(AUDIO_DIR))
)

Audio: 529


In [ ]:
#image embeddings using clip
from transformers import (
    CLIPProcessor,
    CLIPModel
)

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
)

processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

print("CLIP Loaded")

The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP Loaded


In [ ]:
print(type(model))

<class 'transformers.models.clip.modeling_clip.CLIPModel'>


In [ ]:
from PIL import Image
import numpy as np
import torch
import os

count = 0
failed = []

for image_file in os.listdir(FRAME_DIR):

    try:

        image = Image.open(
            os.path.join(
                FRAME_DIR,
                image_file
            )
        ).convert("RGB")

        inputs = processor(
            images=image,
            return_tensors="pt"
        )

        with torch.no_grad():

            output = model.get_image_features(
                **inputs
            )

        # IMPORTANT FIX
        embedding = output.pooler_output

        np.save(

            os.path.join(
                IMAGE_EMB_DIR,
                image_file.replace(
                    ".jpg",
                    ".npy"
                )
            ),

            embedding.cpu().numpy()
        )

        count += 1

    except Exception as e:

        failed.append(
            (image_file, str(e))
        )

print("Saved:", count)
print("Failed:", len(failed))

if failed:
    print(failed[:5])

Saved: 600
Failed: 0


In [ ]:
##verify
import os
import numpy as np

sample = os.listdir(
    IMAGE_EMB_DIR
)[0]

emb = np.load(
    os.path.join(
        IMAGE_EMB_DIR,
        sample
    )
)

print("Shape:", emb.shape)

print(
    "Frames:",
    len(os.listdir(FRAME_DIR))
)

print(
    "Image Embeddings:",
    len(os.listdir(IMAGE_EMB_DIR))
)

Shape: (1, 512)
Frames: 600
Image Embeddings: 600


In [ ]:
#sound enbeddings
!pip install -q librosa soundfile

In [ ]:
import librosa
import numpy as np
import os

count = 0
failed = []

for wav_file in os.listdir(AUDIO_DIR):

    if not wav_file.endswith(".wav"):
        continue

    try:

        audio_path = os.path.join(
            AUDIO_DIR,
            wav_file
        )

        y, sr = librosa.load(
            audio_path,
            sr=16000
        )

        mfcc = librosa.feature.mfcc(
            y=y,
            sr=sr,
            n_mfcc=128
        )

        embedding = np.mean(
            mfcc,
            axis=1
        )

        save_path = os.path.join(
            AUDIO_EMB_DIR,
            wav_file.replace(
                ".wav",
                ".npy"
            )
        )

        np.save(
            save_path,
            embedding
        )

        count += 1

    except Exception as e:

        failed.append(
            (wav_file, str(e))
        )

print("Saved:", count)
print("Failed:", len(failed))

Saved: 529
Failed: 0


In [ ]:
#verify audio embeddings
print(
    "Audio Embeddings:",
    len(os.listdir(AUDIO_EMB_DIR))
)

sample = os.listdir(
    AUDIO_EMB_DIR
)[0]

emb = np.load(
    os.path.join(
        AUDIO_EMB_DIR,
        sample
    )
)

print(emb.shape)

Audio Embeddings: 529
(128,)


In [ ]:
#create common sample list
#image +video embedding
import os

image_ids = {
    f.replace(".npy","")
    for f in os.listdir(
        IMAGE_EMB_DIR
    )
}

audio_ids = {
    f.replace(".npy","")
    for f in os.listdir(
        AUDIO_EMB_DIR
    )
}

common_ids = sorted(
    image_ids.intersection(
        audio_ids
    )
)

print(
    "Common Samples:",
    len(common_ids)
)

Common Samples: 529


In [ ]:
#save common sanmples to csv
import pandas as pd
df = pd.DataFrame({
    "video_id": common_ids
})

csv_path = os.path.join(
    OUTPUT_DIR,
    "common_samples.csv"
)

df.to_csv(
    csv_path,
    index=False
)

print(df.head())


    video_id
0     video1
1    video10
2   video100
3  video1000
4  video1001


In [ ]:
#Build Dataset Index
import pandas as pd
import os

rows = []

for vid in common_ids:

    rows.append({

        "video_id": vid,

        "image_embedding":
        os.path.join(
            IMAGE_EMB_DIR,
            vid + ".npy"
        ),

        "audio_embedding":
        os.path.join(
            AUDIO_EMB_DIR,
            vid + ".npy"
        )

    })

index_df = pd.DataFrame(rows)

index_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "dataset_index.csv"
    ),

    index=False
)

print(index_df.head())

    video_id                                    image_embedding  \
0     video1  /content/drive/MyDrive/multimodal_project/imag...   
1    video10  /content/drive/MyDrive/multimodal_project/imag...   
2   video100  /content/drive/MyDrive/multimodal_project/imag...   
3  video1000  /content/drive/MyDrive/multimodal_project/imag...   
4  video1001  /content/drive/MyDrive/multimodal_project/imag...   

                                     audio_embedding  
0  /content/drive/MyDrive/multimodal_project/audi...  
1  /content/drive/MyDrive/multimodal_project/audi...  
2  /content/drive/MyDrive/multimodal_project/audi...  
3  /content/drive/MyDrive/multimodal_project/audi...  
4  /content/drive/MyDrive/multimodal_project/audi...  


In [ ]:
print(
    "Videos:",
    len(os.listdir(VIDEO_DIR))
)

print(
    "Frames:",
    len(os.listdir(FRAME_DIR))
)

print(
    "Audio:",
    len(os.listdir(AUDIO_DIR))
)

print(
    "Image Embeddings:",
    len(os.listdir(IMAGE_EMB_DIR))
)

print(
    "Audio Embeddings:",
    len(os.listdir(AUDIO_EMB_DIR))
)

print(
    "Usable Samples:",
    len(common_ids)
)

Videos: 600
Frames: 600
Audio: 529
Image Embeddings: 600
Audio Embeddings: 529
Usable Samples: 529


In [ ]:
import json
import os

summary = {
    "videos": len(os.listdir(VIDEO_DIR)),
    "frames": len(os.listdir(FRAME_DIR)),
    "audio": len(os.listdir(AUDIO_DIR)),
    "image_embeddings": len(os.listdir(IMAGE_EMB_DIR)),
    "audio_embeddings": len(os.listdir(AUDIO_EMB_DIR))
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "day1_summary.json"
    ),
    "w"
) as f:
    json.dump(summary, f, indent=4)

print(summary)

{'videos': 600, 'frames': 600, 'audio': 529, 'image_embeddings': 600, 'audio_embeddings': 529}


In [3]:
import os

PROJECT_DIR = "/content/drive/MyDrive/multimodal_project"

os.chdir(PROJECT_DIR)

print(os.getcwd())

/content/drive/MyDrive/multimodal_project


In [4]:
gitignore_content = """
# Dataset
audio/
frames/

# Generated embeddings
image_embeddings/
audio_embeddings/
audio_embeddings_new/
video_embeddings/

# Training outputs
checkpoints/
outputs/

# Numpy arrays
*.npy

# Model files
*.pt
*.pth

# Python cache
__pycache__/
*.pyc

# Jupyter
.ipynb_checkpoints/

# Environment
.env
venv/
.venv/

# OS
.DS_Store
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_content)

print("Created .gitignore")

Created .gitignore


In [5]:
requirements = """
torch
torchvision
transformers
numpy
pandas
scikit-learn
matplotlib
opencv-python
Pillow
librosa
soundfile
tqdm
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("Created requirements.txt")

Created requirements.txt


In [6]:
import os
print(os.getcwd())

/content/drive/MyDrive/multimodal_project
